In [2]:
import json
import os
import pickle
import random
import sys

from typing import Callable, Dict, List, Optional
import haiku as hk
import ase
import ase.io
import jax
import jax.numpy as jnp
import numpy as np
import optax
import yaml


from model.datasets import becs_eps_datasets
from model.utils import (
    create_directory_with_random_name,
    compute_avg_num_neighbors,
)
from model.data_utils import (
    get_atomic_number_table_from_zs,
    compute_average_E0s,
)
from model.predictors import predict_becs_eps
from model.optimizer import optimizer
from model.becs_eps_train import BECS_EPS_train
from model.loss import BecsEpsLoss


from model.becs_eps_model import BECS_EPS_model

jax.config.update("jax_debug_nans", True)
jax.config.update("jax_debug_infs", True)
np.set_printoptions(precision=3, suppress=True)

In [3]:
with open('data/train_eps.yaml') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)

save_dir_name = create_directory_with_random_name(
    os.path.splitext('eps_training')[0]
)

2025-09-20-00-22-eps_training-testy-sophi


In [4]:
with open('data/train_eps.yaml') as f:
    config = yaml.load(f, Loader=yaml.FullLoader)
save_dir_name = '2025-09-20-00-22-eps_training-testy-sophi'

In [7]:
# Save config and code
with open(f"{save_dir_name}/config.yaml", "w") as f:
    yaml.dump(config, f)
with open(f"{save_dir_name}/train.py", "w") as f:
    with open(sys.argv[0]) as g:
        f.write(g.read())
        
train_loader, valid_loader,test_loader, r_max = becs_eps_datasets(
    r_max = config["cutoff"],
    train_path = config["dataset"]["train_path"],
    #valid_path = config["dataset"]["valid_path"],
    #train_num = config["dataset"]["train_num"],
    valid_num = config["dataset"]["valid_num"],
    n_node = config["dataset"]["num_nodes"],
    n_edge = config["dataset"]["num_edges"],
    n_graph = config["dataset"]["num_graphs"],
)

print(len(train_loader.graphs))
print(len(valid_loader.graphs))


model_fn, params, num_message_passing = BECS_EPS_model(
    r_max=r_max,
    atomic_energies_dict={},
    train_graphs=train_loader.graphs,
    initialize_seed=config["model"]["seed"],
    num_species = config["model"]["num_species"],
    use_sc = True,
    graph_net_steps = config["model"]["num_layers"],
    hidden_irreps = config["model"]["internal_irreps"],
    nonlinearities =  {'e': 'swish', 'o': 'tanh'},
    save_dir_name = save_dir_name,
    reload = config["initialization"]['reload'] if 'reload' in config["initialization"] else None,
)
    
print("num_params:", sum(p.size for p in jax.tree_util.tree_leaves(params)))
    
predictor = jax.jit(
    lambda w, g: predict_becs_eps(lambda *x: model_fn(w, *x), g)
)
    
gradient_transform, steps_per_interval, max_num_intervals = optimizer(
    lr = config["training"]["learning_rate"],
    max_num_intervals = config["training"]["max_num_intervals"],
    steps_per_interval = config["training"]["steps_per_interval"],
    # weight_decay = config["training"]["weight_decay"],
)
optimizer_state = gradient_transform.init(params)
print("optimizer num_params:", sum(p.size for p in jax.tree_util.tree_leaves(optimizer_state)))
    
loss_fn = BecsEpsLoss(
    becs_weight = config["training"]["becs_weight"],
    becs_sum_weight = config["training"]["becs_sum_weight"],
    eps_weight = config["training"]["eps_weight"],
)

100%|██████████| 100/100 [00:00<00:00, 6764.24it/s]

1420
100
z_table= AtomicNumberTable: (np.int64(1), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(55), np.int64(56), np.int64(57), np.int64(72), np.int64(73), np.int64(74), np.int64(75), np.int64(76), np.int64(77), np.int64(78), np.int64(79), np.int64(80), np.int64(81), np.int64(82), np.int64(83))
Compute the average number of neighbors: 28.767
Do not normalize the radia


Traceback (most recent call last):
  File "d:\downloads\e3nn\.venv\Lib\site-packages\IPython\core\interactiveshell.py", line 3699, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "C:\Users\sainy\AppData\Local\Temp\ipykernel_37456\1819020827.py", line 23, in <module>
    model_fn, params, num_message_passing = BECS_EPS_model(
                                            ^^^^^^^^^^^^^^^
  File "d:\downloads\e3nn\e3nn-models\model\becs_eps_model.py", line 451, in BECS_EPS_model
    params = jax.jit(model_.init)(
             ^^^^^^^^^^^^^^^^^^^^^
  File "d:\downloads\e3nn\.venv\Lib\site-packages\jax\_src\traceback_util.py", line 180, in reraise_with_filtered_traceback
    return fun(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^
  File "d:\downloads\e3nn\.venv\Lib\site-packages\jax\_src\pjit.py", line 268, in cache_miss
    executable, pgle_profiler, const_args) = _python_pjit_helper(
                                             ^^^^^^^^^^^^^^^^^^^^
  File "d:\d

In [4]:
BECS_EPS_train(
    predictor,
    params,
    optimizer_state,
    train_loader,
    valid_loader,
    #test_loader,
    gradient_transform,
    loss_fn =loss_fn,
    max_num_intervals = max_num_intervals,
    steps_per_interval = steps_per_interval,
    save_dir_name = save_dir_name,
    patience = config["training"]["patience"],
    #ema_decay = config["training"]["ema_decay"],
)
print('training done!')
    

Started training


eval_train:   0%|          | 0/79 [00:00<?, ?it/s]

Computed h_out_irreps: 48x0e+32x0e+24x0e+32x1o+24x2e
h irreps before addition: 104x0e+32x1o+24x2e
Adding self-connection with target irreps: 104x0e+32x1o+24x2e
Computed h_out_irreps: 48x0e+32x0e+32x0e+24x0e+24x0e+32x1o+32x1e+24x2o+24x2e
h irreps before addition: 160x0e+32x1o+32x1e+24x2o+24x2e
Adding self-connection with target irreps: 160x0e+32x1o+32x1e+24x2o+24x2e
Computed h_out_irreps: 48x0e+48x0o+32x0e+32x0e+24x0e+24x0e+32x1o+32x1e+24x2o+24x2e
h irreps before addition: 48x0e+48x0o+112x0e+32x1o+32x1e+24x2o+24x2e
Adding self-connection with target irreps: 48x0e+48x0o+112x0e+32x1o+32x1e+24x2o+24x2e
Compiled function `model` for args:
- n_node=[ 3  8  8  9 10  8  2  8 10 12 10 10 12 12  4 10 12  2 10  2  8 22  0  0] total=192
- n_edge=[112 140 160 166 336 208  32 284 152 152 404 124 352 208  76 156 336  36
 316  52 160  38   0   0] total=4000
cache size: 1


eval_train:   1%|▏         | 1/79 [00:02<03:46,  2.90s/it]

Computed h_out_irreps: 48x0e+32x0e+24x0e+32x1o+24x2e
h irreps before addition: 104x0e+32x1o+24x2e
Adding self-connection with target irreps: 104x0e+32x1o+24x2e
Computed h_out_irreps: 48x0e+32x0e+32x0e+24x0e+24x0e+32x1o+32x1e+24x2o+24x2e
h irreps before addition: 160x0e+32x1o+32x1e+24x2o+24x2e
Adding self-connection with target irreps: 160x0e+32x1o+32x1e+24x2o+24x2e
Computed h_out_irreps: 48x0e+48x0o+32x0e+32x0e+24x0e+24x0e+32x1o+32x1e+24x2o+24x2e
h irreps before addition: 48x0e+48x0o+112x0e+32x1o+32x1e+24x2o+24x2e
Adding self-connection with target irreps: 48x0e+48x0o+112x0e+32x1o+32x1e+24x2o+24x2e


...

Interval 29: eval_valid: mae_becs=3.9784, 
Interval 29: Time per interval: 239.5s, among which 22.5s for evaluation.


eval_train:  99%|█████████▊| 78/79 [00:20<00:00,  3.76it/s]


Interval 30: eval_train: mae_becs=3.6526, 


eval_valid: 100%|██████████| 5/5 [00:01<00:00,  3.67it/s]


Interval 30: eval_valid: mae_becs=4.1085, 
Interval 30: Time per interval: 242.5s, among which 22.2s for evaluation.
Training complete
training done!


In [1]:
model_fn, params, num_message_passing = BECS_EPS_model(
    r_max=r_max,
    atomic_energies_dict={},
    train_graphs=[],
    initialize_seed=config["model"]["seed"],
    num_species = config["model"]["num_species"],
    use_sc = True,
    graph_net_steps = config["model"]["num_layers"],
    hidden_irreps = config["model"]["internal_irreps"],
    nonlinearities =  {'e': 'swish', 'o': 'tanh'},
    save_dir_name = save_dir_name,
    reload = '2024-03-12-18-18-eps_training-prime-lea',
)


NameError: name 'BECS_EPS_model' is not defined

In [6]:
print(len(valid_loader.graphs))

idx = 7

valid_graph = valid_loader.graphs[idx]
# ground truth Born effective charge
print(valid_graph.nodes.becs)

# predicted Born effective charges
pred_becs = predictor(params, valid_graph)
print(pred_becs['becs'])


100
[[[ 1.917  0.     0.   ]
  [ 0.     1.917  0.   ]
  [-0.    -0.     1.917]]

 [[ 1.917  0.     0.   ]
  [ 0.     1.917  0.   ]
  [-0.    -0.     1.917]]

 [[-3.835  0.     0.   ]
  [ 0.    -3.835  0.   ]
  [ 0.     0.    -3.835]]]
[[[-1.956  0.     0.   ]
  [ 0.    -1.956 -0.   ]
  [ 0.    -0.    -1.956]]

 [[-1.956  0.     0.   ]
  [ 0.    -1.956 -0.   ]
  [ 0.    -0.    -1.956]]

 [[ 3.913 -0.    -0.   ]
  [-0.     3.913  0.   ]
  [-0.     0.     3.913]]]


In [7]:


print(valid_graph.globals.eps)

print(pred_becs['eps'])

[[[14.029 -0.    -0.   ]
  [-0.    14.029 -0.   ]
  [-0.    -0.    14.029]]]
[[[11.607 -0.    -0.   ]
  [-0.    11.607  0.   ]
  [-0.     0.    11.607]]]
